Just dumping final dump into silver schema.

No analysis bullshit (manually checking data, etc...)

In [0]:
%sql
create or replace temporary view v_customers_distinct
as
select distinct * from gizmobox.bronze.v_customers
where customer_id is not null;

In [0]:
%sql
create table gizmobox.silver.customers -- managed, not external :)
with cte_max as
(
  select customer_id,max(created_timestamp) as max_created_timestamp
from v_customers_distinct
group by customer_id
)
select
cast(t.created_timestamp as timestamp) as created_timestamp,
t.customer_id,
t.customer_name,
cast(t.date_of_birth as date) as date_of_birth,
t.email,
cast(t.member_since as date) as member_since,
t.telephone
from v_customers_distinct t
join cte_max m
on t.customer_id = m.customer_id
and t.created_timestamp = m.max_created_timestamp

In [0]:
%sql describe extended gizmobox.silver.customers